# Train-only pair discovery

This notebook finds statistically plausible pairs using **only days 1–500**. It excludes ALGO and never looks at days 501–1000.

**Workflow:** take the 20 highest-correlated pairs → keep only initial cointegration survivors → scan for broader alternatives with relaxed thresholds. Notebook 05 decides whether an alternative earns a place using days 501–700.

## Rules and split

- **Discovery (1–500):** top-20 initial screen and later alternative-pair scan.
- **Validation (501–700):** test single candidates and incremental additions in notebook 05.
- **Final test (701–1000):** held for final evaluation after pairs and parameters are frozen.

A selected portfolio may contain each ticker **once only**. Candidate rows may overlap at this stage, but notebook 05 rejects any addition that reuses a ticker already in the book.

In [132]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, coint

DISCOVERY_END = 500
CORRELATION_FLOOR = 0.15  # relatively high positive daily-return correlation
ROLLING_BETA_WINDOW = 120
ROLLING_BETA_STEP = 20

# First-pass statistical screen.
STRICT_EG_PVALUE = 0.10
STRICT_ADF_PVALUE = 0.10
STRICT_MAX_HALF_LIFE = 80
STRICT_MIN_ZERO_CROSSINGS = 8
STRICT_MAX_BETA_CV = 0.60

# Looser review screen: candidates still need validation before use.
RELAXED_EG_PVALUE = 0.15
RELAXED_ADF_PVALUE = 0.10
RELAXED_MAX_HALF_LIFE = 80
RELAXED_MIN_ZERO_CROSSINGS = 8
RELAXED_MAX_BETA_CV = 0.80

TOP_CORRELATED_PAIRS = 20
MAX_VALIDATION_CANDIDATES = 18
EXPLORATORY_PAIRS = [('ULXY', 'HETT')]  # special follow-up from the earlier workflow

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'backtester').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from inside the project directory.')

prices = pd.read_csv(repo_root / 'backtester' / 'data' / '2026' / 'prices.txt', sep=r'\s+', nrows=DISCOVERY_END)
stock_tickers = prices.columns.drop('ALGO')
stock_returns = prices[stock_tickers].pct_change(fill_method=None)
log_prices = np.log(prices[stock_tickers])

assert len(prices) == DISCOVERY_END
assert 'ALGO' not in stock_tickers
print(f'Discovery data: days 1–{DISCOVERY_END}; {len(stock_tickers)} tradable stocks; ALGO excluded.')

Discovery data: days 1–500; 50 tradable stocks; ALGO excluded.


## 1. Initial screen: top 20 return-correlated pairs

This follows your original process: start with the 20 most positively correlated non-ALGO pairs, then test which of those are cointegrated. Correlation narrows the initial universe; it is not proof of a tradable relationship.

In [133]:
return_correlation = stock_returns.corr()

def pair_diagnostics(ticker_a, ticker_b):
    log_a, log_b = log_prices[ticker_a], log_prices[ticker_b]
    coint_stat, eg_pvalue, _ = coint(log_a, log_b)
    beta, intercept = np.polyfit(log_b, log_a, 1)
    spread = log_a - (intercept + beta * log_b)
    adf_pvalue = adfuller(spread, autolag='AIC')[1]

    delta = spread.diff().dropna()
    lagged = spread.shift(1).loc[delta.index]
    speed = np.polyfit(lagged, delta, 1)[0]
    half_life = -np.log(2) / speed if speed < 0 else np.inf

    centred = (spread - spread.mean()).to_numpy()
    zero_crossings = int(np.count_nonzero(np.signbit(centred[:-1]) != np.signbit(centred[1:])))

    rolling_betas = []
    for end in range(ROLLING_BETA_WINDOW, len(spread) + 1, ROLLING_BETA_STEP):
        rolling_betas.append(np.polyfit(log_b.iloc[end - ROLLING_BETA_WINDOW:end], log_a.iloc[end - ROLLING_BETA_WINDOW:end], 1)[0])
    beta_cv = np.std(rolling_betas, ddof=1) / abs(np.mean(rolling_betas)) if len(rolling_betas) > 1 and abs(np.mean(rolling_betas)) > 1e-12 else np.inf

    return {
        'ticker_a': ticker_a, 'ticker_b': ticker_b, 'pair': f'{ticker_a}-{ticker_b}',
        'return_correlation': return_correlation.loc[ticker_a, ticker_b],
        'eg_statistic': coint_stat, 'eg_pvalue': eg_pvalue,
        'hedge_beta': beta, 'intercept': intercept, 'adf_pvalue': adf_pvalue,
        'half_life_days': half_life, 'zero_crossings': zero_crossings, 'beta_cv': beta_cv,
    }

upper_triangle = return_correlation.where(np.triu(np.ones(return_correlation.shape), k=1).astype(bool))
correlation_pairs = upper_triangle.stack().rename('return_correlation').rename_axis(['ticker_a', 'ticker_b']).reset_index().sort_values('return_correlation', ascending=False)
top20_correlated_pairs = correlation_pairs.head(TOP_CORRELATED_PAIRS)

# Fit both A–B and B–A. The same economic pair can produce different
# regression residuals, hedge ratios, and trading signals by orientation.
scan_rows = [pair_diagnostics(left, right) for a, b in combinations(stock_tickers, 2) for left, right in [(a, b), (b, a)]]
broad_scan_df = pd.DataFrame(scan_rows)
broad_scan_df['pair_key'] = broad_scan_df.apply(lambda row: ' | '.join(sorted([row['ticker_a'], row['ticker_b']])), axis=1)
top20_pair_keys = {' | '.join(sorted([row.ticker_a, row.ticker_b])) for row in top20_correlated_pairs.itertuples()}
initial_scan_df = broad_scan_df.loc[broad_scan_df['pair_key'].isin(top20_pair_keys)].copy()

display(
    initial_scan_df.sort_values('return_correlation', ascending=False).style.format({
        'return_correlation': '{:+.3f}', 'eg_pvalue': '{:.4g}', 'adf_pvalue': '{:.4g}',
        'half_life_days': '{:.1f}', 'beta_cv': '{:.3f}', 'hedge_beta': '{:.3f}',
    }).set_caption('Cointegration diagnostics for the initial top-20 correlation screen')
)

,ticker_a,ticker_b,pair,return_correlation,eg_statistic,eg_pvalue,hedge_beta,intercept,adf_pvalue,half_life_days,zero_crossings,beta_cv,pair_key
1172,CUBO,BENI,CUBO-BENI,+0.463,-1.390385,0.8008,-0.737,7.363552,0.588,71.6,18,1.734,BENI | CUBO
1173,BENI,CUBO,BENI-CUBO,+0.463,-3.317673,0.05235,-0.145,3.991240,0.01418,22.8,36,2.347,BENI | CUBO
2200,IHOZ,ILVX,IHOZ-ILVX,+0.455,-1.840911,0.6096,-0.311,6.018216,0.3614,49.1,28,2.926,IHOZ | ILVX
2201,ILVX,IHOZ,ILVX-IHOZ,+0.455,-1.626428,0.7096,-0.679,7.036795,0.47,76.1,10,1.603,IHOZ | ILVX
2385,ILVX,BENI,ILVX-BENI,+0.445,-1.828626,0.6157,-1.670,9.213280,0.3673,52.3,23,1.118,BENI | ILVX
2384,BENI,ILVX,BENI-ILVX,+0.445,-3.392719,0.04317,-0.168,3.906184,0.0113,20.1,33,12.355,BENI | ILVX
270,SRNA,BENI,SRNA-BENI,+0.438,-0.157370,0.9815,1.091,1.147663,0.9444,inf,14,2.047,BENI | SRNA
271,BENI,SRNA,BENI-SRNA,+0.438,-3.135125,0.08146,0.108,2.763928,0.02414,28.2,28,1.839,BENI | SRNA
2093,ITPA,ACIX,ITPA-ACIX,+0.437,-3.864127,0.01111,0.888,0.517164,0.002307,8.3,59,0.291,ACIX | ITPA
2092,ACIX,ITPA,ACIX-ITPA,+0.437,-4.656256,0.0006702,0.967,-0.082347,0.0001037,8.4,75,0.208,ACIX | ITPA


## 2. Initial survivors, then relaxed alternatives

The initial strict screen is applied only to the **top 20** pairs. Its survivors form the initial book; the number is determined by the data, not forced to be three. The broader strict and relaxed scans then examine every pair without a correlation cutoff, so they can surface lower-correlation but strongly cointegrated alternatives.

A lower Engle–Granger/ADF p-value, lower beta CV, shorter half-life, higher correlation, and more zero-crossings all improve the diagnostic score. This score only orders candidates; it is not a trading result.

In [134]:
initial_filter = initial_scan_df['return_correlation'].ge(CORRELATION_FLOOR)
initial_survivors = initial_scan_df.loc[
    initial_filter
    & initial_scan_df['eg_pvalue'].lt(STRICT_EG_PVALUE)
    & initial_scan_df['adf_pvalue'].lt(STRICT_ADF_PVALUE)
    & initial_scan_df['half_life_days'].lt(STRICT_MAX_HALF_LIFE)
    & initial_scan_df['zero_crossings'].ge(STRICT_MIN_ZERO_CROSSINGS)
    & initial_scan_df['beta_cv'].lt(STRICT_MAX_BETA_CV)
].copy()

# The alternative scan intentionally has no correlation filter. Correlation was
# only the initial top-20 narrowing step, matching the original workflow.
base_filter = pd.Series(True, index=broad_scan_df.index)
strict_candidates = broad_scan_df.loc[
    base_filter
    & broad_scan_df['eg_pvalue'].lt(STRICT_EG_PVALUE)
    & broad_scan_df['adf_pvalue'].lt(STRICT_ADF_PVALUE)
    & broad_scan_df['half_life_days'].lt(STRICT_MAX_HALF_LIFE)
    & broad_scan_df['zero_crossings'].ge(STRICT_MIN_ZERO_CROSSINGS)
    & broad_scan_df['beta_cv'].lt(STRICT_MAX_BETA_CV)
].copy()
relaxed_candidates = broad_scan_df.loc[
    base_filter
    & broad_scan_df['eg_pvalue'].lt(RELAXED_EG_PVALUE)
    & broad_scan_df['adf_pvalue'].lt(RELAXED_ADF_PVALUE)
    & broad_scan_df['half_life_days'].lt(RELAXED_MAX_HALF_LIFE)
    & broad_scan_df['zero_crossings'].ge(RELAXED_MIN_ZERO_CROSSINGS)
    & broad_scan_df['beta_cv'].lt(RELAXED_MAX_BETA_CV)
].copy()

def add_quality_score(frame):
    frame = frame.copy()
    if frame.empty:
        return frame.assign(diagnostic_score=pd.Series(dtype=float))
    frame['diagnostic_score'] = (
        frame['return_correlation'].rank(pct=True)
        + (1 - frame['eg_pvalue'].rank(pct=True))
        + (1 - frame['adf_pvalue'].rank(pct=True))
        + (1 - frame['half_life_days'].rank(pct=True))
        + frame['zero_crossings'].rank(pct=True)
        + (1 - frame['beta_cv'].rank(pct=True))
    ) / 6
    return frame.sort_values('diagnostic_score', ascending=False)

def keep_best_orientation(frame):
    return frame.sort_values('diagnostic_score', ascending=False).drop_duplicates('pair_key').copy()

initial_survivors = keep_best_orientation(add_quality_score(initial_survivors))
core_pairs = initial_survivors.copy()
core_pairs['candidate_stage'] = 'core'
strict_candidates = keep_best_orientation(add_quality_score(strict_candidates))
relaxed_candidates = keep_best_orientation(add_quality_score(relaxed_candidates))
relaxed_only = relaxed_candidates.loc[~relaxed_candidates['pair_key'].isin(strict_candidates['pair_key'])].copy()

print(f'Initial top-20 survivors: {len(core_pairs)} | Broad strict candidates: {len(strict_candidates)} | Additional relaxed candidates: {len(relaxed_only)}')
display(core_pairs.style.format({
    'return_correlation': '{:+.3f}', 'eg_pvalue': '{:.4g}', 'adf_pvalue': '{:.4g}',
    'half_life_days': '{:.1f}', 'zero_crossings': '{:.0f}', 'beta_cv': '{:.3f}', 'diagnostic_score': '{:.3f}',
}).set_caption('Initial survivors: top-20 correlation screen followed by strict cointegration diagnostics'))
display(relaxed_only.head(12).style.format({
    'return_correlation': '{:+.3f}', 'eg_pvalue': '{:.4g}', 'adf_pvalue': '{:.4g}',
    'half_life_days': '{:.1f}', 'zero_crossings': '{:.0f}', 'beta_cv': '{:.3f}', 'diagnostic_score': '{:.3f}',
}).set_caption('Relaxed-only candidates — review, not selection'))

Initial top-20 survivors: 2 | Broad strict candidates: 36 | Additional relaxed candidates: 42


,ticker_a,ticker_b,pair,return_correlation,eg_statistic,eg_pvalue,hedge_beta,intercept,adf_pvalue,half_life_days,zero_crossings,beta_cv,pair_key,diagnostic_score,candidate_stage
2448,MHRM,EAFC,MHRM-EAFC,+0.375,-5.193248,7.153e-05,0.941036,-0.479305,9.301e-06,6.7,67,0.355,EAFC | MHRM,0.562,core
2092,ACIX,ITPA,ACIX-ITPA,+0.437,-4.656256,0.0006702,0.967296,-0.082347,0.0001037,8.4,75,0.208,ACIX | ITPA,0.479,core


,ticker_a,ticker_b,pair,return_correlation,eg_statistic,eg_pvalue,hedge_beta,intercept,adf_pvalue,half_life_days,zero_crossings,beta_cv,pair_key,diagnostic_score
1129,NWIG,CUBO,NWIG-CUBO,+0.338,-3.925364,0.009148,-1.649326,12.747915,0.001883,11.5,54,0.620,CUBO | NWIG,0.708
1009,MTNS,MSDP,MTNS-MSDP,+0.228,-4.084700,0.005418,0.265743,2.052681,0.001042,11.0,64,0.667,MSDP | MTNS,0.662
24,AENO,CUBO,AENO-CUBO,+0.372,-3.714381,0.01756,-1.633102,12.601751,0.003966,12.9,55,0.671,AENO | CUBO,0.655
1863,MTNS,ALUT,MTNS-ALUT,+0.112,-4.240237,0.003167,0.330240,2.013973,0.0005711,10.9,58,0.741,ALUT | MTNS,0.583
979,RTTH,MSDP,RTTH-MSDP,+0.232,-3.596085,0.02478,0.539784,1.780277,0.005909,13.5,55,0.753,MSDP | RTTH,0.518
951,BENI,NPCK,BENI-NPCK,+0.330,-3.660714,0.02057,0.163325,2.888202,0.004701,19.0,50,0.707,BENI | NPCK,0.514
1913,NAYO,ACAC,NAYO-ACAC,+0.089,-4.328123,0.002314,0.377903,1.934749,0.000404,9.5,42,0.775,ACAC | NAYO,0.503
1769,NAYO,RRES,NAYO-RRES,+0.413,-3.517220,0.0309,-0.186850,4.214970,0.007655,13.6,36,0.698,NAYO | RRES,0.501
1719,FWWG,AGVF,FWWG-AGVF,+0.234,-3.383856,0.04418,0.483536,1.206990,0.01167,14.5,59,0.778,AGVF | FWWG,0.460
1410,RTTH,ACAC,RTTH-ACAC,+0.102,-3.851278,0.01157,0.498776,1.504104,0.002458,13.5,36,0.657,ACAC | RTTH,0.455


## 3. Freeze initial survivors; export alternative candidates

The initial survivors are the discovery core. Their number comes from the top-20 cointegration screen; it is not set by a target. The remaining high-ranked broad-scan candidates are exported for notebook 05. They are not selected yet.

This is where the no-overlap rule begins: the core itself has no repeated stock. Notebook 05 applies the same rule when considering each addition.

In [135]:
# Keep the best remaining candidates. Overlaps are allowed in this candidate pool;
# notebook 05 will test each candidate only when it does not reuse a selected ticker.
candidate_pool = pd.concat([strict_candidates, relaxed_only], ignore_index=True)
candidate_pool = candidate_pool.loc[~candidate_pool['pair_key'].isin(core_pairs['pair_key'])]
candidate_pool = candidate_pool.drop_duplicates('pair_key').sort_values('diagnostic_score', ascending=False)
validation_candidates = candidate_pool.head(MAX_VALIDATION_CANDIDATES).copy()
# Preserve explicitly motivated exploratory tests even if they rank below the
# generic queue cutoff. ULXY-HETT passed the statistical screen but was a
# dedicated follow-up in the earlier workflow.
exploratory_candidates = pd.DataFrame([pair_diagnostics(a, b) for a, b in EXPLORATORY_PAIRS])
exploratory_candidates['pair_key'] = exploratory_candidates.apply(lambda row: ' | '.join(sorted([row['ticker_a'], row['ticker_b']])), axis=1)
exploratory_candidates['diagnostic_score'] = np.nan
validation_candidates = pd.concat([validation_candidates, exploratory_candidates], ignore_index=True).drop_duplicates('pair_key')
validation_candidates['candidate_stage'] = 'validate_addition'

pair_selection_export = pd.concat([core_pairs, validation_candidates], ignore_index=True)
pair_selection_export['discovery_end_day'] = DISCOVERY_END
export_path = repo_root / 'research_outputs' / 'pair_discovery_candidates.csv'
export_path.parent.mkdir(exist_ok=True)
pair_selection_export.to_csv(export_path, index=False)

if not core_pairs[['ticker_a', 'ticker_b']].stack().is_unique:
    raise ValueError('Initial survivors reuse a ticker. Resolve that conflict explicitly; do not silently discard one.')
display(core_pairs.style.format({'return_correlation': '{:+.3f}', 'eg_pvalue': '{:.4g}', 'adf_pvalue': '{:.4g}', 'half_life_days': '{:.1f}', 'beta_cv': '{:.3f}', 'diagnostic_score': '{:.3f}'}).set_caption('Frozen initial survivors from the top-20 screen'))
display(validation_candidates[['pair', 'return_correlation', 'eg_pvalue', 'adf_pvalue', 'half_life_days', 'zero_crossings', 'beta_cv', 'diagnostic_score']].style.format({'return_correlation': '{:+.3f}', 'eg_pvalue': '{:.4g}', 'adf_pvalue': '{:.4g}', 'half_life_days': '{:.1f}', 'beta_cv': '{:.3f}', 'diagnostic_score': '{:.3f}'}).set_caption('Candidate queue for validation in notebook 05'))
print(f'Exported {len(pair_selection_export)} rows to {export_path.relative_to(repo_root)}')

,ticker_a,ticker_b,pair,return_correlation,eg_statistic,eg_pvalue,hedge_beta,intercept,adf_pvalue,half_life_days,zero_crossings,beta_cv,pair_key,diagnostic_score,candidate_stage
2448,MHRM,EAFC,MHRM-EAFC,+0.375,-5.193248,7.153e-05,0.941036,-0.479305,9.301e-06,6.7,67,0.355,EAFC | MHRM,0.562,core
2092,ACIX,ITPA,ACIX-ITPA,+0.437,-4.656256,0.0006702,0.967296,-0.082347,0.0001037,8.4,75,0.208,ACIX | ITPA,0.479,core


,pair,return_correlation,eg_pvalue,adf_pvalue,half_life_days,zero_crossings,beta_cv,diagnostic_score
0,SMAH-ILVX,+0.299,1.99e-06,2.126e-07,5.2,86,0.306,0.920
1,NWIG-AENO,+0.253,4.043e-06,4.463e-07,4.1,76,0.127,0.894
2,HUXZ-ACAC,+0.250,1.483e-05,1.751e-06,6.0,83,0.172,0.838
3,NGTE-EORC,+0.231,4.688e-06,5.224e-07,5.4,71,0.311,0.766
4,ULXY-HETT,+0.147,9.496e-06,1.095e-06,5.7,75,0.252,0.739
5,EELT-CTGI,+0.246,1.787e-05,2.131e-06,6.1,78,0.391,0.735
6,NWIG-CUBO,+0.338,0.009148,0.001883,11.5,54,0.620,0.708
7,MTNS-MSDP,+0.228,0.005418,0.001042,11.0,64,0.667,0.662
8,AENO-CUBO,+0.372,0.01756,0.003966,12.9,55,0.671,0.655
9,NWIG-DUCT,+0.238,0.004969,0.000945,10.5,73,0.306,0.627


Exported 20 rows to research_outputs/pair_discovery_candidates.csv


## Discovery conclusion

- The core consists only of the survivors of the **top-20 correlation → strict cointegration** screen; it was not chosen to have three pairs.
- Relaxed candidates were kept so strong economic contributors are not discarded solely by a strict cutoff.
- Next: notebook 05 tests candidates on validation days 501–700, accepts only improving non-overlapping additions, then tunes the resulting frozen book.
- Do not use days 701–1000 to change any of these decisions.